# InsightCart — ETL Pipeline

This notebook demonstrates the ETL process for the InsightCart data warehouse.

## ETL Flow

1. Extract data from the source MySQL database (`insightcart`)
2. Transform data using Pandas
3. Perform dimension-key lookups
4. Create the warehouse-ready fact dataset
5. Validate the transformed data
6. Load the data into the MySQL warehouse (`insightcart_dw`)

## Warehouse Model

The warehouse follows a star schema with:

- `dim_customer`
- `dim_product`
- `dim_seller`
- `dim_date`
- `fact_sales`

The Python ETL loads the transformed fact data into `fact_sales_etl` for demonstration and validation.

## 1. Import Libraries

In [1]:
import pandas as pd
import mysql.connector

print("Pandas version:", pd.__version__)
print("MySQL connector imported successfully")

Pandas version: 2.3.0
MySQL connector imported successfully


## 2. Connect to Source Database

In [2]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Priyasaini_16092005",
    database="insightcart"
)

print("Connected to MySQL successfully!")

Connected to MySQL successfully!


## 3. Extract Source Data

In [3]:
customers = pd.read_sql(
    "SELECT * FROM customers",
    conn
)

print("Customers extracted:", len(customers))

C:\Users\abhis\AppData\Local\Temp\ipykernel_30464\2562291921.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers = pd.read_sql(


Customers extracted: 99441


## 3. Extract Source Data

In [4]:
products = pd.read_sql(
    "SELECT * FROM products",
    conn
)

orders = pd.read_sql(
    "SELECT * FROM orders",
    conn
)

order_items = pd.read_sql(
    "SELECT * FROM order_items",
    conn
)

sellers = pd.read_sql(
    "SELECT * FROM sellers",
    conn
)

print("Products:", len(products))
print("Orders:", len(orders))
print("Order items:", len(order_items))
print("Sellers:", len(sellers))

C:\Users\abhis\AppData\Local\Temp\ipykernel_30464\1314605611.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products = pd.read_sql(
C:\Users\abhis\AppData\Local\Temp\ipykernel_30464\1314605611.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders = pd.read_sql(
C:\Users\abhis\AppData\Local\Temp\ipykernel_30464\1314605611.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  order_items = pd.read_sql(


Products: 32951
Orders: 99441
Order items: 112650
Sellers: 3095


C:\Users\abhis\AppData\Local\Temp\ipykernel_30464\1314605611.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sellers = pd.read_sql(


In [5]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [6]:
customers.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [ ]:
## 4. Transform Source Data

In [7]:
customers_clean = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    ]
].copy()

print("Customer columns after transformation:")
print(customers_clean.columns.tolist())

Customer columns after transformation:
['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']


In [8]:
products_clean = products[
    [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].copy()

products_clean = products_clean.rename(
    columns={
        "product_category_name": "category",
        "product_weight_g": "weight",
        "product_length_cm": "length",
        "product_height_cm": "height",
        "product_width_cm": "width"
    }
)

print("Product columns after transformation:")
print(products_clean.columns.tolist())

Product columns after transformation:
['product_id', 'category', 'weight', 'length', 'height', 'width']


In [9]:
orders_clean = orders[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    ]
].copy()

orders_clean["order_purchase_timestamp"] = pd.to_datetime(
    orders_clean["order_purchase_timestamp"]
)

print(orders_clean.dtypes)

order_id                            object
customer_id                         object
order_status                        object
order_purchase_timestamp    datetime64[ns]
dtype: object


In [10]:
order_items_clean = order_items[
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
].copy()

print(order_items_clean.head())

                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id   price  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   58.90   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36  239.90   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d  199.00   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   12.99   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87  199.90   

   freight_value  
0          13.29  
1          19.93  
2          17.87  
3          12.79  
4          18.14  


In [11]:
order_items_clean["quantity"] = 1

print(order_items_clean.head())

                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id   price  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   58.90   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36  239.90   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d  199.00   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   12.99   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87  199.90   

   freight_value  quantity  
0          13.29         1  
1          19.93         1  
2          17.87         1  
3          12.79         1  
4          18.14         1  


In [12]:
order_items_clean["total_amount"] = (
    order_items_clean["price"]
    + order_items_clean["freight_value"]
)

print(order_items_clean.head())

                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id   price  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   58.90   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36  239.90   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d  199.00   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   12.99   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87  199.90   

   freight_value  quantity  total_amount  
0          13.29         1         72.19  
1          19.93         1        259.83  
2          17.87         1        216.87  
3          12.79

## 5. Build Fact Dataset

In [13]:
fact_data = order_items_clean.merge(
    orders_clean[
        [
            "order_id",
            "customer_id",
            "order_purchase_timestamp"
        ]
    ],
    on="order_id",
    how="inner"
)

print(fact_data.head())
print("Rows after merge:", len(fact_data))

                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id   price  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   58.90   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36  239.90   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d  199.00   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   12.99   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87  199.90   

   freight_value  quantity  total_amount                       customer_id  \
0          13.29         1         72.19  3ce436f183e68e07877b285a838db11a   
1          19.93         1      

In [14]:
fact_data["date_key"] = (
    fact_data["order_purchase_timestamp"].dt.year * 10000
    + fact_data["order_purchase_timestamp"].dt.month * 100
    + fact_data["order_purchase_timestamp"].dt.day
)

print(fact_data[[
    "order_purchase_timestamp",
    "date_key"
]].head())

  order_purchase_timestamp  date_key
0      2017-09-13 08:59:02  20170913
1      2017-04-26 10:53:06  20170426
2      2018-01-14 14:33:31  20180114
3      2018-08-08 10:00:35  20180808
4      2017-02-04 13:57:51  20170204


In [15]:
print(fact_data.columns.tolist())

['order_id', 'order_item_id', 'product_id', 'seller_id', 'price', 'freight_value', 'quantity', 'total_amount', 'customer_id', 'order_purchase_timestamp', 'date_key']


In [16]:
dw_conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Priyasaini_16092005",
    database="insightcart_dw"
)

print("Connected to warehouse successfully!")

Connected to warehouse successfully!


## 6. Lookup Dimension Keys

In [17]:
dim_customers = pd.read_sql(
    """
    SELECT customer_key, customer_id
    FROM dim_customer
    """,
    dw_conn
)

print(dim_customers.head())

C:\Users\abhis\AppData\Local\Temp\ipykernel_30464\3549443348.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_customers = pd.read_sql(


   customer_key                       customer_id
0             1  06b8999e2fba1a1fbc88172c00ba8bc7
1             2  18955e83d337fd6b2def6b18a428ac77
2             3  4e7b3e00288586ebd08712fdd0374a03
3             4  b2b6027bc5c5109e529d4dc6358b12c3
4             5  4f2d8ab171c80ec8364f7c12e35b23ad


In [18]:
fact_data = fact_data.merge(
    dim_customers,
    on="customer_id",
    how="left"
)

print(fact_data[[
    "customer_id",
    "customer_key"
]].head())

                        customer_id  customer_key
0  3ce436f183e68e07877b285a838db11a         65558
1  f6dd3ec061db4e3987629fe6b26e5cce         34266
2  6489ae5e4333f3693df5ad4372dab6d3         34956
3  d4eb9395c8c0431ee92fce09860c5a06         51764
4  58dbd0b2d70206bf40e62cd34e84d795          7603


In [19]:
dim_products = pd.read_sql(
    """
    SELECT product_key, product_id
    FROM dim_product
    """,
    dw_conn
)

print(dim_products.head())

C:\Users\abhis\AppData\Local\Temp\ipykernel_30464\218723043.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_products = pd.read_sql(


   product_key                        product_id
0            1  1e9e8ef04dbcff4541ed26657ea517e5
1            2  3aa071139cb16b67ca9e5dea641aaa2f
2            3  96bd76ec8810374ed1b65e291975717f
3            4  cef67bcfe19066a932b7673e239eb23d
4            5  9dc1a7de274444849c219cff195d0b71


In [20]:
fact_data = fact_data.merge(
    dim_products,
    on="product_id",
    how="left"
)

print(fact_data[[
    "product_id",
    "product_key"
]].head())

                         product_id  product_key
0  4244733e06e7ecb4970a6e2683c13e61        25866
1  e5f2d52b802189ee658865ca93d83a8f        27231
2  c777355d18b72b67abbeef9df44fd0fd        22625
3  7634da152a4610f1595efa32f14722fc        15404
4  ac6c3623068f30de03045865e4e10089         8863


In [21]:
dim_sellers = pd.read_sql(
    """
    SELECT seller_key, seller_id
    FROM dim_seller
    """,
    dw_conn
)

print(dim_sellers.head())

   seller_key                         seller_id
0           1  3442f8959a84dea7ee197c632cb2df15
1           2  d1b65fc7debc3361ea86b5f14c68d2e2
2           3  ce3ad9de960102d0677a81f5d0bb7b2d
3           4  c0f3eea2e14555b6faeea3dd58c1b1c3
4           5  51a04a8a6bdcb23deccc82b0b80742cf


C:\Users\abhis\AppData\Local\Temp\ipykernel_30464\1996850203.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_sellers = pd.read_sql(


In [22]:
fact_data = fact_data.merge(
    dim_sellers,
    on="seller_id",
    how="left"
)

print(fact_data[[
    "seller_id",
    "seller_key"
]].head())

                          seller_id  seller_key
0  48436dade18ac8b2bce089ec2a041202         514
1  dd7ddc04e1b6c2c614352b383efe2d36         472
2  5b51032eddd242adc84c38acab88f23d        1825
3  9d7a1d34a5052409006425275ba1c2b4        2024
4  df560393f3a51e74553ab94004ba5c87        1598


## 7. Validate ETL Data

In [23]:
print("Missing customer keys:", fact_data["customer_key"].isna().sum())
print("Missing product keys:", fact_data["product_key"].isna().sum())
print("Missing seller keys:", fact_data["seller_key"].isna().sum())
print("Missing date keys:", fact_data["date_key"].isna().sum())

Missing customer keys: 0
Missing product keys: 0
Missing seller keys: 0
Missing date keys: 0


In [24]:
fact_data_final = fact_data[
    [
        "order_id",
        "date_key",
        "customer_key",
        "product_key",
        "seller_key",
        "quantity",
        "price",
        "freight_value",
        "total_amount"
    ]
].copy()

print(fact_data_final.head())
print(fact_data_final.columns.tolist())

                           order_id  date_key  customer_key  product_key  \
0  00010242fe8c5a6d1ba2dd792cb16214  20170913         65558        25866   
1  00018f77f2f0320c557190d7a144bdd3  20170426         34266        27231   
2  000229ec398224ef6ca0657da4fc703e  20180114         34956        22625   
3  00024acbcdf0a6daa1e931b038114c75  20180808         51764        15404   
4  00042b26cf59d7ce69dfabb4e55b4fd9  20170204          7603         8863   

   seller_key  quantity   price  freight_value  total_amount  
0         514         1   58.90          13.29         72.19  
1         472         1  239.90          19.93        259.83  
2        1825         1  199.00          17.87        216.87  
3        2024         1   12.99          12.79         25.78  
4        1598         1  199.90          18.14        218.04  
['order_id', 'date_key', 'customer_key', 'product_key', 'seller_key', 'quantity', 'price', 'freight_value', 'total_amount']


In [25]:
fact_data_final = fact_data_final.rename(
    columns={
        "price": "product_price"
    }
)

print(fact_data_final.columns.tolist())

['order_id', 'date_key', 'customer_key', 'product_key', 'seller_key', 'quantity', 'product_price', 'freight_value', 'total_amount']


In [26]:
print("Rows:", len(fact_data_final))

print("Missing order IDs:", fact_data_final["order_id"].isna().sum())
print("Missing date keys:", fact_data_final["date_key"].isna().sum())
print("Missing customer keys:", fact_data_final["customer_key"].isna().sum())
print("Missing product keys:", fact_data_final["product_key"].isna().sum())
print("Missing seller keys:", fact_data_final["seller_key"].isna().sum())

print("\nFinal columns:")
print(fact_data_final.columns.tolist())

Rows: 112650
Missing order IDs: 0
Missing date keys: 0
Missing customer keys: 0
Missing product keys: 0
Missing seller keys: 0

Final columns:
['order_id', 'date_key', 'customer_key', 'product_key', 'seller_key', 'quantity', 'product_price', 'freight_value', 'total_amount']


## 8. Load Data into Warehouse

In [27]:
cursor = dw_conn.cursor()

insert_query = """
INSERT INTO fact_sales_etl (
    order_id,
    date_key,
    customer_key,
    product_key,
    seller_key,
    quantity,
    product_price,
    freight_value,
    total_amount
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

for _, row in fact_data_final.iterrows():
    cursor.execute(
        insert_query,
        (
            row["order_id"],
            int(row["date_key"]),
            int(row["customer_key"]),
            int(row["product_key"]),
            int(row["seller_key"]),
            int(row["quantity"]),
            float(row["product_price"]),
            float(row["freight_value"]),
            float(row["total_amount"])
        )
    )

dw_conn.commit()

print("Data loaded successfully!")

Data loaded successfully!


In [28]:
check = pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM fact_sales_etl",
    dw_conn
)

print(check)

   total_rows
0      112650


C:\Users\abhis\AppData\Local\Temp\ipykernel_30464\4152086372.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  check = pd.read_sql(


In [29]:
dw_conn.close()

print("Warehouse connection closed.")

Warehouse connection closed.


In [30]:
dw_conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Priyasaini_16092005",
    database="insightcart_dw"
)

print("Connected to warehouse successfully!")

Connected to warehouse successfully!


In [31]:
cursor = dw_conn.cursor()

insert_query = """
INSERT INTO fact_sales_etl (
    order_id,
    date_key,
    customer_key,
    product_key,
    seller_key,
    quantity,
    product_price,
    freight_value,
    total_amount
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

data_to_insert = [
    (
        row["order_id"],
        int(row["date_key"]),
        int(row["customer_key"]),
        int(row["product_key"]),
        int(row["seller_key"]),
        int(row["quantity"]),
        float(row["product_price"]),
        float(row["freight_value"]),
        float(row["total_amount"])
    )
    for _, row in fact_data_final.iterrows()
]

cursor.executemany(insert_query, data_to_insert)
dw_conn.commit()

print("Rows loaded:", cursor.rowcount)

Rows loaded: 112650


In [32]:
print("fact_data rows:", len(fact_data))
print("fact_data_final rows:", len(fact_data_final))

fact_data rows: 112650
fact_data_final rows: 112650


In [33]:
print("Customer duplicate IDs:",
      dim_customers["customer_id"].duplicated().sum())

print("Product duplicate IDs:",
      dim_products["product_id"].duplicated().sum())

print("Seller duplicate IDs:",
      dim_sellers["seller_id"].duplicated().sum())

Customer duplicate IDs: 0
Product duplicate IDs: 0
Seller duplicate IDs: 0


In [34]:
print("customers:", len(customers))
print("products:", len(products))
print("orders:", len(orders))
print("order_items:", len(order_items))

print("\nCleaned data:")
print("customers_clean:", len(customers_clean))
print("products_clean:", len(products_clean))
print("orders_clean:", len(orders_clean))
print("order_items_clean:", len(order_items_clean))

print("\nCurrent fact data:")
print("fact_data:", len(fact_data))
print("fact_data_final:", len(fact_data_final))

customers: 99441
products: 32951
orders: 99441
order_items: 112650

Cleaned data:
customers_clean: 99441
products_clean: 32951
orders_clean: 99441
order_items_clean: 112650

Current fact data:
fact_data: 112650
fact_data_final: 112650


In [35]:
dw_conn.close()

print("Warehouse connection closed.")

Warehouse connection closed.


## 10. ETL Summary

The InsightCart ETL pipeline successfully:

- Extracted source data from MySQL.
- Transformed the data using Pandas.
- Created derived fields such as `quantity`, `total_amount`, and `date_key`.
- Performed dimension-key lookups for customers, products, and sellers.
- Validated the transformed data.
- Loaded 112,650 fact records into the MySQL warehouse ETL table.
- Performed post-load validation to confirm data consistency.

The resulting dataset is structured for analysis using the InsightCart star-schema data warehouse.